# 01 — External Data Preparation

**Project:** SERSA Export Activity and Industrial Consumption Dynamics  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

This notebook prepares the two data sources that feed the entire project:

1. **External data:** Monthly automotive parts trade flows (Exportaciones + Importaciones) from Mexico's Secretaría de Economía — DataMéxico API, covering HS4 code 8708 (parts and accessories for motor vehicles).

2. **Internal data:** Vending machine transaction records from `master_sales.csv`, the output of the main project *SERSA Industrial Bajío — Vending Machine Sales Analysis*.

### Data sources

| Dataset | File | Coverage | Rows |
|---|---|---|---|
| Trade flows | Raw CSV from DataMéxico API | Jan 2006 – Feb 2026 | 484 |
| Sales transactions | `master_sales.csv` | Apr 2022 – May 2026 | 639,568 |

### Overlap period
Both datasets share **46 months** of concurrent data: **April 2022 – February 2026**.  
All analysis in subsequent notebooks uses this overlap period exclusively.

### Goals of this notebook
1. Load and inspect both raw datasets.
2. Separate trade flows into Exportaciones and Importaciones series.
3. Aggregate sales transactions into monthly totals.
4. Align both datasets to the same monthly frequency (Month Start).
5. Trim to the overlap period.
6. Export `merged_monthly_data.csv` as the shared input for notebooks 02–05.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

---
## 2. Configuration

In [ ]:
# Paths — run from within notebooks/
RAW_DIR       = os.path.join(os.getcwd(), "..", "data", "raw")
PROCESSED_DIR = os.path.join(os.getcwd(), "..", "data", "processed")
OUTPUT_TABLES = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_FIGURES= os.path.join(os.getcwd(), "..", "outputs", "figures")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_TABLES,  exist_ok=True)
os.makedirs(OUTPUT_FIGURES, exist_ok=True)

# Trade flow file — long original name from DataMéxico
TRADE_FILE = "Comercio-internacional-neto-de-Partes-y-Accesorios-de-Vehiculos-Automotores--Flujo-mensual-segun-datos-de-Cinta-de-Aduanas.csv"
SALES_FILE = "master_sales.csv"

print(f"Raw dir:        {os.path.normpath(RAW_DIR)}")
print(f"Processed dir:  {os.path.normpath(PROCESSED_DIR)}")
print(f"Output tables:  {os.path.normpath(OUTPUT_TABLES)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")

---
## 3. Load Trade Flow Data

In [ ]:
trade_raw = pd.read_csv(os.path.join(RAW_DIR, TRADE_FILE))

print(f"Shape: {trade_raw.shape}")
print(f"Columns: {trade_raw.columns.tolist()}")
print(f"Flows: {trade_raw['Flow'].unique().tolist()}")
print(f"Nulls: {trade_raw.isnull().sum().sum()}")
print()
print(trade_raw.head(6).to_string())

---
## 4. Process Trade Flows

Steps:
- Parse `Month` column to datetime (format `YYYY-MM`).
- Separate into Exportaciones and Importaciones.
- Convert Trade Value from USD to millions USD for readability.
- Rename columns for clarity.

In [ ]:
# Parse month
trade_raw["Month"] = pd.to_datetime(trade_raw["Month"], format="%Y-%m")

# Convert to millions USD
trade_raw["Trade_Value_MUSD"] = (trade_raw["Trade Value"] / 1_000_000).round(2)

# Separate flows
exports = (
    trade_raw[trade_raw["Flow"] == "Exportaciones"]
    [["Month", "Trade_Value_MUSD"]]
    .rename(columns={"Trade_Value_MUSD": "exports_musd"})
    .sort_values("Month")
    .reset_index(drop=True)
)

imports = (
    trade_raw[trade_raw["Flow"] == "Importaciones"]
    [["Month", "Trade_Value_MUSD"]]
    .rename(columns={"Trade_Value_MUSD": "imports_musd"})
    .sort_values("Month")
    .reset_index(drop=True)
)

print(f"Exportaciones: {len(exports)} months  |  {exports['Month'].min().date()} -> {exports['Month'].max().date()}")
print(f"Importaciones: {len(imports)} months  |  {imports['Month'].min().date()} -> {imports['Month'].max().date()}")
print()
print(exports.head(3).to_string())

---
## 5. Load and Aggregate Sales Data

In [ ]:
sales_raw = pd.read_csv(os.path.join(RAW_DIR, SALES_FILE))
sales_raw["Fecha de Consumo"] = pd.to_datetime(sales_raw["Fecha de Consumo"])

print(f"Sales rows: {len(sales_raw):,}")
print(f"Date range: {sales_raw['Fecha de Consumo'].min().date()} -> {sales_raw['Fecha de Consumo'].max().date()}")
print()
print(sales_raw.head(3).to_string())

In [ ]:
# Aggregate to monthly totals
monthly_sales = (
    sales_raw
    .groupby(pd.Grouper(key="Fecha de Consumo", freq="MS"))
    .agg(
        transactions = ("SKU", "count"),
        revenue_mxn  = ("Price", "sum"),
        profit_mxn   = ("Profit", "sum")
    )
    .reset_index()
    .rename(columns={"Fecha de Consumo": "Month"})
)

monthly_sales["revenue_mxn"] = monthly_sales["revenue_mxn"].round(2)
monthly_sales["profit_mxn"]  = monthly_sales["profit_mxn"].round(2)

print(f"Monthly sales: {len(monthly_sales)} months")
print()
print(monthly_sales.head(5).to_string())

---
## 6. Merge and Trim to Overlap Period

The overlap between trade data and sales data runs from **April 2022 to February 2026** — 46 months.

In [ ]:
# Merge trade flows with sales
merged = (
    exports
    .merge(imports, on="Month", how="inner")
    .merge(monthly_sales, on="Month", how="inner")
    .sort_values("Month")
    .reset_index(drop=True)
)

print(f"Merged shape: {merged.shape}")
print(f"Overlap period: {merged['Month'].min().date()} -> {merged['Month'].max().date()}")
print(f"Months: {len(merged)}")
print()
print(merged.head(5).to_string())

---
## 7. Data Quality Check

In [ ]:
print("Nulls per column:")
print(merged.isnull().sum())
print()
print("Summary statistics:")
print(merged.describe().round(2).to_string())

---
## 8. Overview Plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Exports
axes[0].plot(merged["Month"], merged["exports_musd"], color="#2C7BB6", linewidth=2)
axes[0].set_ylabel("Exports (MUSD)", fontsize=10)
axes[0].set_title("Automotive Parts Exports — Mexico (HS4 8708)", fontsize=12, fontweight="bold")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
axes[0].grid(axis="y", alpha=0.3)
sns.despine(ax=axes[0])

# Imports
axes[1].plot(merged["Month"], merged["imports_musd"], color="#D7191C", linewidth=2)
axes[1].set_ylabel("Imports (MUSD)", fontsize=10)
axes[1].set_title("Automotive Parts Imports — Mexico (HS4 8708)", fontsize=12, fontweight="bold")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
axes[1].grid(axis="y", alpha=0.3)
sns.despine(ax=axes[1])

# Sales
axes[2].plot(merged["Month"], merged["revenue_mxn"], color="#1A9641", linewidth=2)
axes[2].set_ylabel("Revenue (MXN)", fontsize=10)
axes[2].set_title("SERSA Monthly Revenue — All Clients", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Month", fontsize=10)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[2].grid(axis="y", alpha=0.3)
sns.despine(ax=axes[2])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "01_overview_three_series.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 01_overview_three_series.png")

---
## 9. Export

In [ ]:
# Main merged dataset
merged.to_csv(
    os.path.join(PROCESSED_DIR, "merged_monthly_data.csv"),
    index=False,
    decimal=","
)

# Monthly sales standalone
monthly_sales.to_csv(
    os.path.join(PROCESSED_DIR, "monthly_sales.csv"),
    index=False,
    decimal=","
)

# Monthly exports standalone
exports.to_csv(
    os.path.join(PROCESSED_DIR, "monthly_exports.csv"),
    index=False,
    decimal=","
)

# Monthly imports standalone
imports.to_csv(
    os.path.join(PROCESSED_DIR, "monthly_imports.csv"),
    index=False,
    decimal=","
)

print("Export complete.")
print(f"  merged_monthly_data.csv  ->  {merged.shape[0]} months x {merged.shape[1]} columns")
print(f"  monthly_sales.csv        ->  {len(monthly_sales)} months")
print(f"  monthly_exports.csv      ->  {len(exports)} months")
print(f"  monthly_imports.csv      ->  {len(imports)} months")

---
## 10. Summary

| Output | Description |
|--------|-------------|
| `merged_monthly_data.csv` | 46 months × 6 columns — main analytical dataset |
| `monthly_sales.csv` | Monthly transactions, revenue, profit |
| `monthly_exports.csv` | Monthly automotive parts exports (MUSD) |
| `monthly_imports.csv` | Monthly automotive parts imports (MUSD) |
| `01_overview_three_series.png` | Three-panel time series overview |

**Columns in `merged_monthly_data.csv`:**

| Column | Description |
|--------|-------------|
| `Month` | Month Start date (YYYY-MM-DD) |
| `exports_musd` | Automotive parts exports — Mexico (millions USD) |
| `imports_musd` | Automotive parts imports — Mexico (millions USD) |
| `transactions` | Total vending machine dispense events |
| `revenue_mxn` | Total revenue (MXN) |
| `profit_mxn` | Total profit (MXN) |

**Next:** `02_trend_relationships.ipynb` — visual and statistical examination of whether trade flows and sales move together over the overlap period.